# Practice Notebook — Informal (Non-Bank) Lending Market Analysis

**Based on:** *Python for Engineering and Scientific Computing*, Chapter 3 (NumPy) —
the same vectorized-array, `np.linalg.solve()`, and statistical-function toolkit
used for the runtime comparison, mesh-network, and lightning-protection examples,
applied here to the **informal lending sector** (moneylenders, rotating savings
groups, short-term cash lenders operating outside the formal banking system).

This is a topic studied extensively in development and financial-inclusion
economics. The goal of this notebook is **analytical and protective**: to
quantify the true cost of informal credit and the risk profile of informal
lending, the same way a researcher, regulator, or financial-literacy program
would — **not** to optimize how to extract more money from borrowers.

| Part | Question | Chapter technique |
|---|---|---|
| 1 | How expensive is a short-term informal loan, really? | Vectorized arrays (Section 3.1.1, Listing 3.2) |
| 2 | How should a small lender mix loan products to meet its own funding obligations? | Linear systems, `np.linalg.solve()` (Section 3.4) |
| 3 | How much default risk does an informal loan portfolio carry? | Monte Carlo + statistics (Section 3.1.5) |

## Learning objectives
1. Convert a short-term "flat fee" loan quote into an **effective annual rate
   (EAR)** using vectorized array math, and see why short terms make quoted
   rates misleading
2. Build a repayment matrix from a word problem and solve `A·x = b` for a
   loan-product mix that meets a fixed obligation, exactly like the chapter's
   mesh-current and cash-flow-matching examples
3. Run a Monte Carlo simulation of loan defaults across a portfolio and
   summarize the risk with `np.mean`, `np.std`, `np.percentile`, and
   `np.where`
4. Reason about **when this kind of model is, and is not, appropriate to
   use** — and about the real-world ethical and legal issues around informal
   lending that no notebook can substitute for

## How to use this notebook
- Every task cell contains a `# TODO` and hints. Replace `None` / `...` with
  working code.
- Run cells top to bottom — later tasks depend on variables created earlier.
- Each task has an `assert` sanity check directly below it.
- Don't peek at the cheat sheet until you've tried the task yourself.
- Section 8 ("Limitations") has no code — it is short-answer reflection, and
  it is the most important section in this particular notebook.


## Setup

In [ ]:
import numpy as np
from numpy.linalg import solve

np.set_printoptions(precision=2, suppress=True)


---
## Part 1 — The true annualized cost of a short-term loan (vectorized)

Informal lenders very often quote a **flat fee per loan cycle** rather than an
annual rate — for example, "pay back 10% more, whenever the loan is due."
That sounds small, but if the loan only lasts a few weeks, the *equivalent
annual rate* (what you'd pay if you kept re-borrowing all year) is enormous.
This is the same idea used to critique payday lending in many countries.

For a loan with flat fee rate `fee_rate` and a term of `t` weeks, the number of
times that cycle repeats in a year is `52 / t`, and the effective annual rate
(EAR), compounding each cycle, is:

`EAR = (1 + fee_rate) ** (52 / t) - 1`

Rather than a loop over each term, build this as **arrays** (Listing 3.1/3.5
style): one array of terms, one array of cycles-per-year, one array of EARs —
all computed in a few vectorized lines.

### Task 1
**TODO:**
1. `terms_weeks` — every loan term from 1 to 8 weeks (`np.arange`)
2. `cycles_per_year` — `52 / terms_weeks` (vectorized division)
3. `EAR` — the formula above, applied to the whole `cycles_per_year` array at
   once (no loop)

In [ ]:
fee_rate = 0.10  # 10% flat fee per loan cycle

terms_weeks = None       # TODO: np.arange(1, 9)
cycles_per_year = None   # TODO
EAR = None                # TODO

# --- sanity check ---
assert terms_weeks.shape == (8,)
assert EAR.shape == terms_weeks.shape
for t, ear in zip(terms_weeks, EAR):
    print(f"{t}-week loan, {fee_rate*100:.0f}% flat fee -> EAR = {ear*100:8.1f}%")


---
## Part 2 — Loan product mix to match a funding obligation (linear system)

A small informal lender doesn't lend out its own money forever — it typically
draws capital from a rotating savings pool or a family/community backer, and
owes that capital (with its own return) back on a schedule. To meet that
schedule, the lender chooses how much to lend out through each of its loan
products, each with a different term and rate.

This is a **cash-flow-matching** problem, structurally identical to the
mesh-current network (Table 3.2) and retirement bond-ladder examples:
**`A·x = b`**.

Three loan products, each with **flat interest on the original principal**,
repaid in equal level installments over the term:

| Product | Term (months) | Flat monthly rate |
|---|---|---|
| A | 1 | 10% |
| B | 2 | 8% |
| C | 3 | 6% |

For $1 lent through a product with term `m` and monthly flat rate `r`, the
lender collects a level payment of `1/m + r` in **every month the loan is
still active** (months `1` through `m`), and nothing after it matures.

The lender needs to collect exactly **\$5,000** in month 1, **\$4,000** in
month 2, and **\$3,000** in month 3, to repay its own backer on schedule.

### Task 2 — Build the repayment matrix `A`
**TODO:** `A[i-1, j]` = the payment collected in month `i` (1, 2, or 3) per
$1 lent through product `j` (0, 1, or 2). A double loop over months and
products is the clearest way to build this: for each `(i, j)`, if
`i <= terms[j]`, the payment is `1/terms[j] + rates[j]`; otherwise it's 0.

In [ ]:
terms = np.array([1, 2, 3])
rates = np.array([0.10, 0.08, 0.06])
n = len(terms)

A = np.zeros((n, n))
# TODO: fill in A using two nested loops over month i (1..3) and product j (0..2)
#   if i <= terms[j]: A[i-1, j] = 1/terms[j] + rates[j]

# --- sanity check ---
assert A.shape == (3, 3)
assert A[2, 0] == 0  # product A (1-month term) has already matured by month 3
print("Repayment matrix A:\n", A)


### Task 3 — Build the obligation vector `b`
**TODO:** `b` is the amount the lender must collect each month, in month
order.

In [ ]:
b = None  # TODO: np.array([...]) — months 1, 2, 3 collection targets

# --- sanity check ---
assert b.shape == (3,)
print("Required monthly collections b:", b)


### Task 4 — Solve for the lending amount per product
**TODO:** Solve `A x = b` for `x`, then verify `A @ x` reproduces `b`.

In [ ]:
x = None  # TODO: solve(A, b)

# --- sanity check ---
assert x.shape == (3,)
check = None  # TODO: A @ x  (should closely match b)
print("Amount to lend via each product:", np.round(x, 2))
print("Check A @ x == b:", np.round(check, 2))
print("Total capital deployed: $%.2f" % np.sum(x))


---
## Part 3 — Monte Carlo default risk

Informal loans are usually made without collateral, credit checks, or legal
enforcement — so default risk is real and needs to be priced in, not
ignored. Simulate a portfolio of **200 loans**, each \$100 principal, each
carrying a 10% flat fee (so a performing loan repays \$110). Each loan has a
**15% chance of default**, and defaulted loans still recover **30%** of
principal (e.g., through partial repayment or informal social pressure).

### Task 5 — Simulate defaults across 2,000 scenarios
**TODO:**
1. `np.random.seed(1)` for reproducibility.
2. `defaults` — a `(2000, 200)` boolean array: `True` where a loan defaults.
   Use `np.random.random((n_scenarios, n_loans)) < default_prob`.
3. `collections_per_loan` — for each loan in each scenario, the amount
   collected: `defaulted_payoff` if `defaults` is `True`, otherwise
   `performing_payoff`. Use `np.where(condition, value_if_true, value_if_false)`.

In [ ]:
np.random.seed(1)
n_loans = 200
principal = 100
flat_rate = 0.10
default_prob = 0.15
recovery_rate = 0.30
n_scenarios = 2000

performing_payoff = principal * (1 + flat_rate)
defaulted_payoff = principal * recovery_rate

defaults = None              # TODO
collections_per_loan = None  # TODO: np.where(defaults, defaulted_payoff, performing_payoff)

# --- sanity check ---
assert defaults.shape == (n_scenarios, n_loans)
assert collections_per_loan.shape == (n_scenarios, n_loans)
print("Simulation array built.")


### Task 6 — Total collections per scenario
**TODO:** Sum each scenario's per-loan collections into one number per
scenario: `total_collections`, an array of length `n_scenarios`. Use
`np.sum(..., axis=1)` to sum across loans (columns) within each scenario
(row).

In [ ]:
total_collections = None  # TODO: np.sum(collections_per_loan, axis=1)

# --- sanity check ---
assert total_collections.shape == (n_scenarios,)
print("First 5 scenario totals:", np.round(total_collections[:5], 2))


### Task 7 — Summarize the risk
**TODO:** Using the statistical functions from Section 3.1.5, compute:
- `mean_collections`, `std_collections` — `np.mean`, `np.std`
- `p10`, `p90` — 10th and 90th percentile of `total_collections`
  (`np.percentile`)
- `prob_loss` — the fraction of scenarios where `total_collections` is less
  than the total principal lent (`n_loans * principal`) — i.e., the
  probability the portfolio loses money overall (`np.mean` on a boolean
  comparison gives you a fraction directly)
- `prob_cant_repay_backer` — the fraction of scenarios where
  `total_collections` is less than `backer_obligation` (given below)

In [ ]:
total_principal_lent = n_loans * principal
backer_obligation = 21000  # what the lender must repay its own capital source this cycle

mean_collections = None  # TODO
std_collections = None    # TODO
p10 = None                  # TODO
p90 = None                  # TODO

prob_loss = None                  # TODO
prob_cant_repay_backer = None    # TODO

print(f"Mean collections....: ${mean_collections:,.2f}")
print(f"Std deviation........: ${std_collections:,.2f}")
print(f"10th percentile......: ${p10:,.2f}")
print(f"90th percentile......: ${p90:,.2f}")
print(f"P(portfolio loses money): {prob_loss*100:.1f}%")
print(f"P(can't repay backer's ${backer_obligation:,}): {prob_cant_repay_backer*100:.1f}%")


---
## Section 8 — Limitations (reflection, no code)

This is the most important section in this notebook. Informal lending is a
real, widely studied part of the financial system, and this model is a
**simplified, purely mathematical** view of it. Before drawing any
conclusion from this notebook, answer the questions below in your own words.

**When is this kind of analysis a reasonable choice?**
- Academic or policy study of *why* informal loan APRs look so extreme
  (Part 1), which is a well-established critique used by regulators and
  consumer-protection researchers, not a novel or fringe idea
- Understanding, at a conceptual level, why short-term unsecured lending
  needs either high pricing or strong loss controls to be financially
  viable (Part 3's finding that a 10% flat fee with realistic default rates
  can still lose money is a genuine, important insight)
- Financial literacy: helping a borrower understand what an informal loan
  actually costs them annualized, before they take it
- A rough teaching example of cash-flow matching (Part 2), the same
  technique used legitimately in bond laddering and pension funding

**When is it NOT recommended — and where this notebook stops on purpose**
- **This notebook does not, and should not, help anyone design loan terms,
  collection practices, or a lending operation meant to extract maximum
  interest or exploit borrowers.** Nothing here recommends interest rates,
  collection tactics, or ways to increase leverage over borrowers, and it
  should not be extended to do so.
- **Usury laws and consumer-protection regulations vary enormously by
  jurisdiction** and are not modeled here at all. Real informal or formal
  lending at rates like those in Part 1 is illegal in many places. This
  notebook computes a rate; it says nothing about whether charging that
  rate is lawful where you are.
- **Informal lending often carries real human costs** — over-indebtedness,
  coercive or violent collection practices, and cycles of debt bondage are
  well-documented risks in under-regulated informal credit markets. A
  spreadsheet-level default/recovery model cannot capture, justify, or
  excuse those harms, and shouldn't be read as doing so.
- **The default probability, recovery rate, and flat fees in Part 3 are
  illustrative assumptions**, not measurements of any real market. Do not
  quote the 15%/30%/82% figures from this notebook as real-world facts.
- **This is not guidance for borrowers or lenders to act on.** A real
  borrower deciding whether to take an informal loan, or a real community
  organization designing a lending program, needs local legal advice and
  real data — not a teaching notebook.

1. In Part 1, why does a *shorter* loan term produce a *higher* EAR for the
   same flat fee? Walk through the math in your own words.
2. In Part 3, the portfolio's mean collections came out *below* the total
   principal lent even with a positive 10% fee on performing loans. What
   two model inputs would you change, and in which direction, to make the
   portfolio breakeven — and what would each of those changes mean in the
   real world for borrowers?
3. Name one real-world protection (legal, regulatory, or community-based)
   that exists specifically because informal lending can be exploitative,
   and explain in one sentence why a Monte Carlo model like this one cannot
   substitute for it.
